# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WingManOO7/ML-INTER/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** The card asks for a top-10 review; the skeleton's top-20 is the optional deeper version.

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — same slice, same windows, same label as ML-04

**Lane check (lanes lock this week): I confirm Lane 2 — Refresh / Content Opportunity Scoring.**

Everything is reused from my ML-04 data contract, so this baseline is scored on the same pages and
label the Week-5 model will use:

- data: warehouse `month=2026-03` partition + `dim_content` (`content_created_date` only); June 2026 stays sealed;
- decision moment **T = Mon 2026-03-16**; feature window 03-02 → 03-15; outcome window 03-16 → 03-29;
- pages: ≥ 100 impressions before T, tracked from 03-02, with a valid position;
- label `declined_next_2w` = impressions in the outcome window more than 20% below the feature window.

I keep two separate frames. **`pages`** holds only what is known at T, and it is the only thing the rule
ever sees. **`labels`** holds the outcome. It is used for two jobs only: testing the signals in
section 1 and scoring the rule in hindsight in section 4.

In [1]:
import os, sys, subprocess, importlib.util, json
from pathlib import Path

if importlib.util.find_spec('duckdb') is None:      # Colab ships duckdb; this covers fresh local envs
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files
from sklearn.model_selection import GroupShuffleSplit

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a cell.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ')

REPO = 'FlyRank/internship-warehouse'
DEV_MONTH = '2026-03'          # mid-panel development month; 2026-06 is the sealed test month

def fetch(path):
    # downloads once into the HF cache; later runs reuse the local copy
    return Path(hf_hub_download(REPO, path, repo_type='dataset', token=HF_TOKEN)).as_posix()

month_files = [f for f in list_repo_files(REPO, repo_type='dataset', token=HF_TOKEN)
               if f.startswith(f'fact_content_daily_performance/month={DEV_MONTH}/')]
con = duckdb.connect()
con.execute('SET enable_progress_bar = false')
con.execute(f"CREATE OR REPLACE VIEW daily AS SELECT * FROM read_parquet({[fetch(f) for f in month_files]})")
con.execute(f"CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('{fetch('dim_content.parquet')}')")
sql = lambda q: con.sql(q).df()

T = pd.Timestamp('2026-03-16')
FEAT_START, FEAT_END = pd.Timestamp('2026-03-02'), pd.Timestamp('2026-03-15')
OUT_START,  OUT_END  = pd.Timestamp('2026-03-16'), pd.Timestamp('2026-03-29')
KEYS, LABEL = ['client_hash_id', 'content_hash_id'], 'declined_next_2w'

# ---- pages: ONLY the feature window (before T) ----
pages = sql(f'''
    SELECT client_hash_id, content_hash_id,
           MIN(report_date)                                                    AS first_seen,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE)      AS imp_pre,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE
                                        AND report_date <  DATE '2026-03-09') AS imp_wk1,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE
                                        AND report_date >= DATE '2026-03-09') AS imp_wk2,
           MAX(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE
                                        AND report_date <  DATE '2026-03-09') AS max_day_wk1,
           SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)
             / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE
                                                   AND gsc_avg_position > 0), 0) AS pos_pre,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)                  AS active_days_pre
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START.date()}' AND DATE '{FEAT_END.date()}'
    GROUP BY ALL
    HAVING imp_pre >= 100
''').merge(sql('SELECT content_hash_id, content_created_date, content_type FROM dim_content'),
           on='content_hash_id', how='left')

pages = pages[pd.to_datetime(pages['first_seen']) == FEAT_START].dropna(subset=['pos_pre']).reset_index(drop=True)
pages[['imp_wk1', 'imp_wk2', 'max_day_wk1']] = pages[['imp_wk1', 'imp_wk2', 'max_day_wk1']].fillna(0)
pages['age_days_at_T']   = (T - pd.to_datetime(pages['content_created_date'])).dt.days
pages['slide_ratio']     = (pages['imp_wk2'] + 1) / (pages['imp_wk1'] + 1)
pages['spike_share_wk1'] = pages['max_day_wk1'] / pages['imp_wk1'].where(pages['imp_wk1'] > 0)
site = pages.groupby('client_hash_id')[['imp_wk1', 'imp_wk2']].sum()
pages['client_slide_ratio'] = pages['client_hash_id'].map((site['imp_wk2'] + 1) / (site['imp_wk1'] + 1))
assert (pages['age_days_at_T'] > 0).all()

# ---- labels: ONLY the outcome window (after T) — signal tests + hindsight scoring, never rule input ----
outcome = sql(f'''
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS imp_out
    FROM daily
    WHERE report_date BETWEEN DATE '{OUT_START.date()}' AND DATE '{OUT_END.date()}'
    GROUP BY ALL
''')
labels = pages[KEYS + ['imp_pre']].merge(outcome, on=KEYS, how='left')
labels[LABEL] = (labels['imp_out'].fillna(0) < 0.8 * labels['imp_pre']).astype(int)
labels = labels[KEYS + [LABEL]]
audit = pages.merge(labels, on=KEYS)              # for checks and hindsight only

print(f"{len(pages):,} pages from {pages['client_hash_id'].nunique()} clients; base rate {labels[LABEL].mean():.3f}")

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'work' / 'notebooks').is_dir()), Path.cwd())
OUT_DIR = ROOT / 'work' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ml04 = OUT_DIR / 'w03_data_contract_receipts.json'
if ml04.exists():                                  # continuity with the ML-04 contract, when run inside the repo
    frame04 = json.loads(ml04.read_text())['frame']
    assert (frame04['pages'], frame04['base_rate']) == (len(pages), round(float(labels[LABEL].mean()), 3))
    print('same pages and base rate as the ML-04 contract')

74,542 pages from 36 clients; base rate 0.337
same pages and base rate as the ML-04 contract


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My starting idea (before any check):** *a page that has gone a long time without a refresh, and
whose impressions are already slipping, should be reviewed first.* It leans on two signals:

- **Signal A — staleness**, the signal behind FlyRank's **refresh flags** (old content gets flagged
  for a refresh). I measure it as `age_days_at_T` = T − `content_created_date`. I can't use the
  update date: ML-04 showed it is a July snapshot that already lies after T for 82.6% of pages.
- **Signal B — recent slide**, the lane guide's "declining with demand" idea:
  `slide_ratio` = (impressions in week 2 + 1) / (impressions in week 1 + 1), both weeks before T.

Each check is one bucket table: `n` pages per bucket, and the share of them that went on to drop by
more than 20% over the next two weeks. The outcome only **tests** the signal here; it never enters
the rule.

### Signal check A — staleness (flag-linked: refresh flags)
If the refresh-flag logic holds, older buckets should decline more.

In [2]:
def bucket_table(col, bins, names):
    b = pd.cut(audit[col], bins=bins, labels=names, right=False)
    return audit.groupby(b, observed=True)[LABEL].agg(n='size', decline_rate='mean').round(3)

age_table = bucket_table('age_days_at_T', [0, 30, 90, 180, 365, np.inf],
                         ['< 30 days', '30–90 days', '90–180 days', '180–365 days', '365+ days'])
print(f'base rate {audit[LABEL].mean():.3f}\n')
print(age_table, '\n')

# does "older declines more" hold INSIDE each big client? (age buckets also mix clients)
big = audit.groupby('client_hash_id').filter(lambda g: len(g) >= 500 and g['age_days_at_T'].nunique() > 1)
gap = big.groupby('client_hash_id').apply(
    lambda g: g.loc[g['age_days_at_T'] >= g['age_days_at_T'].median(), LABEL].mean()
              - g.loc[g['age_days_at_T'] <  g['age_days_at_T'].median(), LABEL].mean(), include_groups=False).dropna()
print(f'clients with >= 500 pages: {len(gap)}; older half declines MORE in {(gap > 0).sum()}, LESS in {(gap < 0).sum()}')

base rate 0.337

                   n  decline_rate
age_days_at_T                     
< 30 days       4742         0.298
30–90 days     17506         0.359
90–180 days    14982         0.463
180–365 days   29714         0.285
365+ days       7598         0.262 



clients with >= 500 pages: 17; older half declines MORE in 8, LESS in 9


**Verdict A — staleness: MIXED.** The decline rate rises from 0.298 (< 30 days) to 0.463 (90–180 days), then falls to 0.285 (180–365 days) and 0.262 (365+ days). Both of those are below the 0.337 base rate. The oldest pages, the ones a refresh flag would target first, decline *least*. Inside the 17 clients with ≥ 500 pages there is no consistent direction: the older half declines more in 8 clients and less in 9. Part of the hump is probably *which sites joined when*, because age buckets mix clients. So page age alone does not predict the next two weeks' drop in this slice. That does not mean old pages never need a refresh. It means age is not a reason to rank one page above another here, so it stays out of my rule.

### Signal check B — recent slide
If slides continue, pages that already lost impressions from week 1 to week 2 should decline more.

In [3]:
slide_table = bucket_table('slide_ratio', [0, 0.5, 0.8, 1.25, 2, np.inf],
                           ['collapsed (< 0.5x)', 'down 20–50%', 'about flat', 'up 25–100%', 'up > 2x'])
print(f'base rate {audit[LABEL].mean():.3f}\n')
slide_table

base rate 0.337



,n,decline_rate
slide_ratio,,
collapsed (< 0.5x),6488,0.625
down 20–50%,16821,0.467
about flat,30392,0.310
up 25–100%,13556,0.195
up > 2x,7285,0.155


**Verdict B — recent slide: CONFIRMED.** The decline rate falls step by step across the buckets: 0.625 for pages that collapsed below half in week 2, 0.467 for down 20–50%, 0.310 for about flat, 0.195 for up 25–100%, and 0.155 for up more than 2×. Every bucket holds thousands of pages (n = 6,488 to 30,392). Pages already sliding before T go on to decline more often. One caveat comes back in section 4: some of this "slide" is a one-day spike in week 1 dropping out, and the label sees that as a decline too.

### The rule, after the checks

Staleness came back MIXED, so **it is out of the rule**. Without the check I would have boosted old
pages that in this slice decline *less* often than average. The rule keeps only the CONFIRMED
signal:

> **A page goes into the refresh-review queue when its impressions fell by more than 20% from week 1
> to week 2 before T. Pages in the queue are ordered by how many weekly impressions they lost, so the
> biggest visible slides come first.**

| Part | Value |
|---|---|
| Score | `imp_wk1 − imp_wk2` when `slide_ratio < 0.8`, else 0. No fitted weights. |
| Reason code (one) | `SLIDING_WITH_DEMAND` |
| Action label | `REFRESH_REVIEW` for flagged pages; every other page gets `MONITOR` |
| Inputs | `imp_wk1`, `imp_wk2`: both from the feature window, before T |

Ordering by lost impressions is a value choice, not a prediction: an editor's hour is worth more on a
page that lost 50,000 impressions than on one that lost 50.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
RULE_INPUTS = ['imp_wk1', 'imp_wk2']                    # both from the feature window, before T

def baseline_rule(df):
    slide = (df['imp_wk2'] + 1) / (df['imp_wk1'] + 1)
    flagged = slide < 0.8
    return pd.DataFrame({'score': np.where(flagged, df['imp_wk1'] - df['imp_wk2'], 0),
                         'reason_code': np.where(flagged, 'SLIDING_WITH_DEMAND', ''),
                         'action': np.where(flagged, 'REFRESH_REVIEW', 'MONITOR')}, index=df.index)

queue = pages[KEYS + ['content_type', 'imp_pre', 'imp_wk1', 'imp_wk2', 'pos_pre', 'active_days_pre', 'age_days_at_T']] \
    .join(baseline_rule(pages[RULE_INPUTS]))
queue = queue.sort_values(['score', 'imp_pre'], ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', np.arange(1, len(queue) + 1))

assert not {LABEL, 'imp_out'} & set(queue.columns), 'the queue must not carry outcome columns'
queue.to_csv(OUT_DIR / 'baseline_action_score.csv', index=False)

print(f"wrote {(OUT_DIR / 'baseline_action_score.csv').relative_to(ROOT)}: {len(queue):,} ranked pages")
print(queue['action'].value_counts().to_string(), '\n')
queue.drop(columns=KEYS).head(5)

wrote work\outputs\baseline_action_score.csv: 74,542 ranked pages
action
MONITOR           51233
REFRESH_REVIEW    23309 



,rank,content_type,imp_pre,imp_wk1,imp_wk2,pos_pre,active_days_pre,age_days_at_T,score,reason_code,action
0,1,keyword article,54825.0,54799.0,26.0,0.159672,14,228,54773.0,SLIDING_WITH_DEMAND,REFRESH_REVIEW
1,2,keyword article,72659.0,63388.0,9271.0,2.955339,14,263,54117.0,SLIDING_WITH_DEMAND,REFRESH_REVIEW
2,3,keyword article,49032.0,42274.0,6758.0,8.435899,14,263,35516.0,SLIDING_WITH_DEMAND,REFRESH_REVIEW
3,4,keyword article,79570.0,54599.0,24971.0,4.411587,14,328,29628.0,SLIDING_WITH_DEMAND,REFRESH_REVIEW
4,5,keyword article,36489.0,30478.0,6011.0,2.205788,14,263,24467.0,SLIDING_WITH_DEMAND,REFRESH_REVIEW


## 3. Top-10 review

*For each of the top 10: the action, why it's there, and what would make it wrong.*

Everything in this table was known at T. Three columns are there only to help me review, and the rule
does not use them:
- `spike_share_wk1`: the share of week-1 impressions that came from the page's single biggest day. A
  steady page sits near 1/7 ≈ 0.14.
- `site_slide`: the whole site's week-2 / week-1 impressions.
- `pos_pre`: the page's average position before T.

Site and page IDs are cut down from the pseudonyms, for reading only.

In [5]:
top10 = queue.head(10).merge(pages[KEYS + ['spike_share_wk1', 'client_slide_ratio']], on=KEYS)
top10_view = top10.assign(site=top10['client_hash_id'].str.replace('client_', '').str[:6],
                          page=top10['content_hash_id'].str.replace('content_', '').str[:6],
                          site_slide=top10['client_slide_ratio'].round(3),
                          spike_share_wk1=top10['spike_share_wk1'].round(2),
                          pos_pre=top10['pos_pre'].round(1))
top10_view[['rank', 'site', 'page', 'action', 'reason_code', 'imp_wk1', 'imp_wk2', 'score',
            'pos_pre', 'active_days_pre', 'spike_share_wk1', 'site_slide', 'age_days_at_T']].set_index('rank')

,site,page,action,reason_code,imp_wk1,imp_wk2,score,pos_pre,active_days_pre,spike_share_wk1,site_slide,age_days_at_T
rank,,,,,,,,,,,,
1,73cda7,9c057b,REFRESH_REVIEW,SLIDING_WITH_DEMAND,54799.0,26.0,54773.0,0.2,14,0.53,0.926,228
2,62f4a7,34a70f,REFRESH_REVIEW,SLIDING_WITH_DEMAND,63388.0,9271.0,54117.0,3.0,14,0.62,0.799,263
3,62f4a7,945d6f,REFRESH_REVIEW,SLIDING_WITH_DEMAND,42274.0,6758.0,35516.0,8.4,14,0.88,0.799,263
4,08a6a7,e7b5dd,REFRESH_REVIEW,SLIDING_WITH_DEMAND,54599.0,24971.0,29628.0,4.4,14,0.23,0.811,328
5,62f4a7,ed50f7,REFRESH_REVIEW,SLIDING_WITH_DEMAND,30478.0,6011.0,24467.0,2.2,14,0.54,0.799,263
6,73cda7,6a9c79,REFRESH_REVIEW,SLIDING_WITH_DEMAND,32270.0,7998.0,24272.0,2.1,14,0.68,0.926,304
7,62f4a7,0c5606,REFRESH_REVIEW,SLIDING_WITH_DEMAND,24335.0,3361.0,20974.0,4.4,14,0.57,0.799,263
8,62f4a7,1642f3,REFRESH_REVIEW,SLIDING_WITH_DEMAND,34061.0,18082.0,15979.0,4.7,14,0.72,0.799,263
9,e547b8,693799,REFRESH_REVIEW,SLIDING_WITH_DEMAND,16946.0,2459.0,14487.0,2.6,12,0.81,1.505,262


One line per pick, using only what was known at T:

| # | Action | Why it's there | What would make it wrong |
|---|---|---|---|
| 1 | REFRESH_REVIEW | Lost 54,773 weekly impressions: week 2 had 26 against 54,799 in week 1, a near-total collapse | A drop to almost zero usually means removal, a redirect or de-indexing, not slow decay. That needs a technical check, not a rewrite. Its position (0.2) is impossible, so the position data can't be trusted either |
| 2 | REFRESH_REVIEW | Lost 54,117 (63,388 → 9,271) while sitting at position 3.0 on page one | One day held 62% of week 1, so this may be a spike returning to normal. Its whole site also slid 20% (0.799), so the cause may be the site, not the page |
| 3 | REFRESH_REVIEW | Lost 35,516 (42,274 → 6,758) at position 8.4, the bottom of page one | 88% of week 1 came from a single day: almost certainly a one-off burst. Same site-wide slide as #2 |
| 4 | REFRESH_REVIEW | Lost 29,628 (54,599 → 24,971, −54%) at position 4.4, with no spike (largest day 23%) | The cleanest pick of the ten. Wrong if this is a one-week dip: two weeks of history can't tell a dip from a trend |
| 5 | REFRESH_REVIEW | Lost 24,467 (30,478 → 6,011) at position 2.2 | Week-1 spike (54% on one day), and the same site-wide slide as #2 |
| 6 | REFRESH_REVIEW | Lost 24,272 (32,270 → 7,998) at position 2.1 | Week-1 spike (68%). Its site was steady (0.926), so the spike, not the site, is the risk |
| 7 | REFRESH_REVIEW | Lost 20,974 (24,335 → 3,361) at position 4.4 | Week-1 spike (57%) plus the site-wide slide. The fourth page from the same site so far |
| 8 | REFRESH_REVIEW | Lost 15,979 (34,061 → 18,082, −47%) at position 4.7 | Week-1 spike (72%) plus the site-wide slide. Five slots from one site look like one site event, not five page problems |
| 9 | REFRESH_REVIEW | Lost 14,487 (16,946 → 2,459) at position 2.6 | Week-1 spike (81%), and only 12 of 14 days had data, so part of the drop may be gaps. In its favour: its site *grew* 1.5×, so the fall looks page-specific |
| 10 | REFRESH_REVIEW | Lost 14,221 (22,251 → 8,030) at position 3.6 | Week-1 spike (76%). The sixth page from the same site, which slid 20% overall |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Three warning signs, all knowable at T, applied to the top 10:
- **week-1 spike**: one day held more than half of week 1's impressions, so the "slide" may just be the
  page returning to normal after a spike;
- **site-wide slide**: the whole site lost ≥ 20% from week 1 to week 2, so the cause may be the site
  (tracking, a search update, seasonality) rather than this page;
- **impossible position**: an average position below 1 cannot come from Search Console, where rank 1
  is the top. It is a measurement artifact in the release, so this page's position can't be trusted.

Then, **in hindsight only**, I score the rule against the outcome window: precision@K next to the base
rate, on all pages and on the same held-out clients as ML-04 (`GroupShuffleSplit`, seed 42).

In [6]:
weak = top10_view[['rank', 'site', 'page']].assign(
    week1_spike=top10['spike_share_wk1'] > 0.5,
    site_wide_slide=top10['client_slide_ratio'] < 0.8,
    impossible_position=top10['pos_pre'] < 1,
)
weak['warnings'] = weak[['week1_spike', 'site_wide_slide', 'impossible_position']].sum(axis=1)
# hindsight, evaluation only: the label, and the weekly level after T compared with the page's week-2 level
weak['hindsight_declined'] = top10.merge(labels, on=KEYS)[LABEL].values
imp_out_top10 = top10.merge(outcome, on=KEYS, how='left')['imp_out'].fillna(0).values
weak['hindsight_week_vs_wk2'] = (imp_out_top10 / 2 / top10['imp_wk2']).round(2).values
print(weak.set_index('rank').to_string(), '\n')
rose = weak[(weak['hindsight_declined'] == 1) & (weak['hindsight_week_vs_wk2'] > 1)]
print(f"top-10 pages with >= 1 warning: {(weak['warnings'] > 0).sum()}; with a week-1 spike: {weak['week1_spike'].sum()}")
print(f"label says declined: {weak['hindsight_declined'].sum()} of 10; of those, weekly impressions after T ROSE above the week-2 level: "
      f"{len(rose)} (ranks {', '.join(map(str, rose['rank']))})\n")

# ---- hindsight scoring (outcome window used for evaluation only) ----
ranked = queue.merge(labels, on=KEYS)
_, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(pages, groups=pages['client_hash_id']))
holdout_clients = set(pages.iloc[test_idx]['client_hash_id'])
held = ranked[ranked['client_hash_id'].isin(holdout_clients)]

def precision_at(df, k):
    return round(float(df.sort_values('rank')[LABEL].head(k).mean()), 3)

KS = [10, 20, 50, 100]
hindsight = pd.DataFrame({
    'all pages':            [len(ranked), round(float(ranked[LABEL].mean()), 3)] + [precision_at(ranked, k) for k in KS],
    'ML-04 holdout clients': [len(held),  round(float(held[LABEL].mean()), 3)]  + [precision_at(held, k) for k in KS],
}, index=['pages', 'base rate'] + [f'precision@{k}' for k in KS])
print(hindsight.to_string(), '\n')

# ---- leakage check ----
feature_window_cols = {'imp_pre', 'imp_wk1', 'imp_wk2', 'max_day_wk1', 'pos_pre', 'active_days_pre',
                       'age_days_at_T', 'slide_ratio', 'spike_share_wk1', 'client_slide_ratio'}
assert set(RULE_INPUTS) <= feature_window_cols, 'rule inputs must come from before T'
assert not {LABEL, 'imp_out'} & set(pages.columns), 'the frame the rule sees holds no outcome column'
product_flags = {'health_score', 'priority_score', 'action_type', 'needs_ctr_fix', 'is_quick_win', 'refresh_tier'}
all_columns = set(sql('DESCRIBE daily')['column_name']) | set(sql('DESCRIBE dim_content')['column_name'])
assert not product_flags & all_columns, 'no product decision flags exist in the release, so none can leak'
assert queue.equals(queue.sort_values('rank')), 'queue order is the rule order'
print('leakage check passed: rule inputs = feature window only; no outcome column in the rule frame; no product flags in the data')

        site    page  week1_spike  site_wide_slide  impossible_position  warnings  hindsight_declined  hindsight_week_vs_wk2
rank                                                                                                                        
1     73cda7  9c057b         True            False                 True         2                   1                   1.00
2     62f4a7  34a70f         True             True                False         2                   0                   3.62
3     62f4a7  945d6f         True             True                False         2                   1                   0.65
4     08a6a7  e7b5dd        False            False                False         0                   0                   2.03
5     62f4a7  ed50f7         True             True                False         2                   1                   1.61
6     73cda7  6a9c79         True            False                False         1                   1                   1.67


               all pages  ML-04 holdout clients
pages          74542.000              36786.000
base rate          0.337                  0.361
precision@10       0.700                  0.800
precision@20       0.650                  0.650
precision@50       0.620                  0.600
precision@100      0.540                  0.570 

leakage check passed: rule inputs = feature window only; no outcome column in the rule frame; no product flags in the data


**Weak picks: 9 of my top 10 carry at least one warning. Only #4 is clean.**
- **Week-1 spikes are the main problem: 9 of 10.** Ranking by *lost impressions* rewards the biggest week-1 numbers, and those are often one-day bursts. The rule cannot tell "fell back after a spike" from "started to decay".
- **One site fills six slots** (`62f4a7`), and that whole site lost 20% from week 1 to week 2. That is more likely one site event than six separate page problems. The six pages also share the same age, 263 days. That looks like a bulk import date rather than each page's real publish date, which is one more reason page age was a weak staleness signal.
- **#1 has an impossible position** (0.2) and an almost empty week 2. That pattern points to removal or de-indexing, a technical check rather than a refresh.

**Hindsight (evaluation only, outcome window).** The rule beats the base rate. On all pages: precision@10 = 0.70, @50 = 0.62, against 0.337. On the ML-04 holdout clients: 0.80 and 0.60, against 0.361. But the top 10 tells a less flattering story. The label calls 7 of them "declined", yet for 4 of those 7 (#5, #6, #7, #9) the weekly impressions after T **rose above the week-2 level**. Their "decline" is the week-1 spike leaving the comparison, not the page decaying. **The label shares the rule's spike blind spot**, so these hindsight numbers flatter spike-heavy picks. Before Week 5 I will build both the slide and the label on a spike-robust measure, for example by dropping each page's biggest day or using median daily impressions.

**Leakage check: passed.** The rule's only inputs are `imp_wk1` and `imp_wk2`, both from the feature window before T. The frame the rule sees has no outcome column. The release ships no product decision flags (`health_score`, `priority_score`, `needs_ctr_fix` and the rest), so none can leak. The CSV carries no label.

**The bar for Week 5:** precision@50 = **0.60** on the ML-04 holdout clients (base rate 0.361), with the same pages, the same label and the same split. This baseline stays frozen from here on.

In [7]:
# Receipts: this run's numbers as JSON (committed); the CSV itself stays out of git.
receipts = {
    'notebook': 'work/notebooks/w04_baseline_score.ipynb',
    'release': 'FlyRank/internship-warehouse v20260703', 'dev_month': DEV_MONTH,
    'decision_moment_T': str(T.date()),
    'feature_window': [str(FEAT_START.date()), str(FEAT_END.date())],
    'outcome_window': [str(OUT_START.date()), str(OUT_END.date())],
    'pages': len(pages), 'base_rate': round(float(labels[LABEL].mean()), 3),
    'signal_A_staleness': {'verdict': 'MIXED', 'table': age_table.reset_index().astype(str).to_dict(orient='records')},
    'signal_B_recent_slide': {'verdict': 'CONFIRMED', 'table': slide_table.reset_index().astype(str).to_dict(orient='records')},
    'rule': {'inputs': RULE_INPUTS, 'flag': 'slide_ratio < 0.8', 'score': 'imp_wk1 - imp_wk2 if flagged else 0',
             'reason_code': 'SLIDING_WITH_DEMAND', 'action': 'REFRESH_REVIEW / MONITOR'},
    'queue': {'rows': len(queue), 'flagged': int((queue['action'] == 'REFRESH_REVIEW').sum()),
              'csv': 'work/outputs/baseline_action_score.csv (not committed)'},
    'hindsight': hindsight.to_dict(),
    'top10_warnings': int((weak['warnings'] > 0).sum()), 'top10_declined': int(weak['hindsight_declined'].sum()),
    'top10_declined_but_rose_above_wk2': rose['rank'].tolist(),
    'holdout_split': 'GroupShuffleSplit(test_size=0.3, random_state=42) on client_hash_id',
}
(OUT_DIR / 'w04_baseline_metrics.json').write_text(json.dumps(receipts, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print('wrote', (OUT_DIR / 'w04_baseline_metrics.json').relative_to(ROOT))

wrote work\outputs\w04_baseline_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.